# Grupo 5 — validação sanitária da base de ouro

Este notebook verifica nulos, duplicatas, datas e regularidade em dias úteis. A variável de previsão é `VALUE`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'validacao_bases.py').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Não foi possível localizar validacao_bases.py.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from validacao_bases import (
    CONFIGURACOES, calcular_sha256, carregar_base, extrair_datas,
    relatorios_como_dataframe, validar_base,
)

NOME_BASE = 'ouro'
config = CONFIGURACOES[NOME_BASE]
dados = carregar_base(NOME_BASE, RAIZ)
print(f'Base: {NOME_BASE} | formato: {dados.shape[0]:,} linhas x {dados.shape[1]} colunas')

Base: ouro | formato: 12,009 linhas x 2 colunas


In [2]:
display(dados.head())
display(dados.dtypes.rename('tipo').to_frame())

,DATE,VALUE
0,1968-04-01,38.000
1,1968-04-02,37.600
2,1968-04-03,37.700
3,1968-04-04,36.700
4,1968-04-05,37.200


,tipo
DATE,object
VALUE,object


## Resultado consolidado

A frequência esperada é de dias úteis (`B`), então sábados e domingos não são tratados como lacunas.

In [3]:
relatorio = validar_base(NOME_BASE, RAIZ)
display(relatorios_como_dataframe({NOME_BASE: relatorio}))
display(pd.Series(relatorio.nulos_por_coluna, name='quantidade_de_nulos').to_frame())

,nome,linhas,colunas,linhas_com_nulos,duplicatas_exatas,datas_invalidas,datas_duplicadas,ordenacao_datas,frequencia_esperada,frequencia_regular,timestamps_ausentes,timestamps_fora_da_grade,aprovada
0,ouro,12009,2,0,0,0,0,crescente,B,True,0,0,True


,quantidade_de_nulos


In [4]:
datas = extrair_datas(dados, config)
problemas = dados.loc[dados.duplicated(keep=False) | datas.duplicated(keep=False)].copy()
problemas.insert(0, 'data_normalizada', datas.loc[problemas.index])
print(f'Linhas com duplicidade exata ou temporal: {len(problemas):,}')
display(problemas.head(10))
print('Exemplos de dias úteis ausentes:', relatorio.exemplos_timestamps_ausentes)
print('Ordenação encontrada:', relatorio.ordenacao_datas)

Linhas com duplicidade exata ou temporal: 0


,data_normalizada,DATE,VALUE


Exemplos de dias úteis ausentes: []
Ordenação encontrada: crescente


## Consolidação na granularidade semanal

Para a modelagem do grupo 5, cada semana é encerrada na sexta-feira (`W-FRI`) e recebe o último preço disponível da semana.

In [ ]:
ouro_temporal = dados.assign(data=datas).drop_duplicates().set_index('data').sort_index()
ouro_semanal = ouro_temporal[['VALUE']].resample(config.frequencia_modelagem).last().dropna()
print(f'Série semanal: {ouro_semanal.shape[0]:,} linhas x {ouro_semanal.shape[1]} coluna')
display(ouro_semanal.head())

## Evidência para o congelamento

O hash identifica exatamente o arquivo analisado. O congelamento completo das cinco bases deve ser feito uma única vez com `congelar_bases('dados_congelados/v1')`.

In [5]:
arquivo = RAIZ / config.caminho
print('Arquivo:', arquivo.relative_to(RAIZ))
print('SHA-256:', calcular_sha256(arquivo))
print('Conclusão:', 'APROVADA' if relatorio.aprovada else 'REQUER TRATAMENTO ANTES DA MODELAGEM')

Arquivo: grupo5\gold.daily.prices.csv
SHA-256: e4a83611fee61bdb70d3707f5044691105828d2b8f8644a88a6e40d2707a2814
Conclusão: APROVADA
